<a href="https://colab.research.google.com/github/ranjani-cse/RAG_Demo/blob/main/RAG_Examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Run this first
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 35.5 MB/s eta 0:00:00


In [10]:
# Step 1: Install required libraries (run once)
!pip install sentence-transformers faiss-cpu

# Step 2: Import and prepare
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# Documents
documents = [
    "The capital of France is Paris.",
    "The Eiffel Tower is in Paris.",
    "India has 28 states.",
    "The capital of India is New Delhi.",
    "Python is a programming language.",
    "The Earth is the third planet from the sun.",
    "Water freezes at 0 degrees Celsius.",
    "The fastest land animal is the cheetah.",
    "Brazil is famous for football"
]

# Load model and create embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(documents)

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

# Query function
def query_rag(query, top_k=2):
    print(f"\n🔍 Question: '{query}'")
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding.astype('float32'), top_k)
    for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        print(f"  {i+1}. {documents[idx]} (distance: {dist:.4f})")
    return [documents[idx] for idx in indices[0]]

def query_rag_with_threshold(query, top_k=2, threshold=1.0):
    """
    Query RAG with a distance threshold.
    If the closest document is too far (distance > threshold),
    return None (meaning "I don't have this information").
    """
    print(f"\n🔍 Question: '{query}'")

    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding.astype('float32'), top_k)

    # If the closest document is too far, return None
    if distances[0][0] > threshold:
        print(f"  ❌ No relevant document (distance: {distances[0][0]:.4f} > {threshold})")
        return None

    print(f"\n📄 Top {top_k} most relevant documents:")
    for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        print(f"  {i+1}. {documents[idx]} (distance: {dist:.4f})")

    return [documents[idx] for idx in indices[0]]

# 15 test questions
test_queries = [
    # Related questions (should work)
    "What is the freezing point of water?",
    "What is the capital of India?",
    "What language is Python?",
    "How many states does India have?",
    "what is the fastest animal?"
    # Unrelated questions (should return "I don't know")
    "What is the population of Singapore?",
    "Who is the Prime Minister of India?",
    "What is the weather today?",
    "What is the meaning of life?",
    "How does photosynthesis work?",
    "What is the square root of 144?",
    "Who won the World Cup?",
    "which country is famous for football?"
]

print("="*60)
print("📝 TESTING RAG WITH 15 UNRELATED QUESTIONS")
print("="*60)

for q in test_queries:
    result = query_rag_with_threshold(q, threshold=1.0)
    if result is None:
        print("  ✅ I don't have this information in my knowledge base.")
    else:
        print(f"  ✅ Retrieved: {result[0][:50]}...")
    print("-"*40)




Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📝 TESTING RAG WITH 15 UNRELATED QUESTIONS

🔍 Question: 'What is the freezing point of water?'

📄 Top 2 most relevant documents:
  1. Water freezes at 0 degrees Celsius. (distance: 0.6191)
  2. The capital of France is Paris. (distance: 1.6460)
  ✅ Retrieved: Water freezes at 0 degrees Celsius....
----------------------------------------

🔍 Question: 'What is the capital of India?'

📄 Top 2 most relevant documents:
  1. The capital of India is New Delhi. (distance: 0.6129)
  2. India has 28 states. (distance: 0.8534)
  ✅ Retrieved: The capital of India is New Delhi....
----------------------------------------

🔍 Question: 'What language is Python?'

📄 Top 2 most relevant documents:
  1. Python is a programming language. (distance: 0.2340)
  2. The capital of France is Paris. (distance: 1.8061)
  ✅ Retrieved: Python is a programming language....
----------------------------------------

🔍 Question: 'How many states does India have?'

📄 Top 2 most relevant documents:
  1. India has 28 sta